# LeetCode 1039 — Minimum Score Triangulation of Polygon bằng Python

Link đề: https://leetcode.com/problems/minimum-score-triangulation-of-polygon/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        ...
```

Ghi chú: có một đa giác **lồi** `n` đỉnh, đỉnh thứ `i` mang giá trị `values[i]` (theo chiều kim đồng hồ). **Tam giác phân** (triangulation) là chia đa giác thành `n - 2` tam giác mà đỉnh của mỗi tam giác đều là đỉnh của đa giác. **Trọng số** một tam giác là **tích** ba giá trị ở đỉnh; **điểm** của một tam giác phân là tổng trọng số của các tam giác. Tìm điểm **nhỏ nhất**. Đây là bài **quy hoạch động trên đoạn** (interval DP) kinh điển: cạnh `(i, j)` của đa giác con thuộc đúng **một** tam giác `(i, k, j)`, chia bài toán thành hai đa giác con nhỏ hơn. Một cách "tham lam" quen thuộc (cắt đỉnh nhỏ nhất) **sai**; ta sẽ thấy phản ví dụ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tam giác phân đa giác lồi, trọng số là tích ba đỉnh, mục tiêu tổng nhỏ nhất.
2. Nắm các công cụ Python cần thiết: bảng 2 chiều, vòng lặp theo **độ dài đoạn**, `functools.lru_cache`, `itertools.combinations` (sinh tập đường chéo), và kiểm tra các đoạn thẳng có cắt nhau.
3. Hiểu nguyên lý **quy hoạch động đoạn**: `dp[i][j]` là điểm nhỏ nhất của đa giác con gồm các đỉnh `i, i + 1, ..., j` (cạnh `(i, j)` nối hai đầu); cạnh `(i, j)` thuộc một tam giác duy nhất `(i, k, j)` với `i < k < j`, nên `dp[i][j] = min_k dp[i][k] + dp[k][j] + v[i] * v[k] * v[j]`.
4. Đi qua 4 cách: sinh mọi tam giác phân bằng tập đường chéo không cắt nhau và tính điểm (brute force, oracle), đệ quy không nhớ theo cùng công thức (số Catalan), đệ quy có nhớ, và bảng bottom-up theo độ dài (cách nộp chính).
5. Tránh các lỗi kinh điển: **tham lam** cắt đỉnh nhỏ nhất, **thứ tự lặp** sai làm dùng ô chưa tính, khởi tạo `dp` bằng `0` rồi lấy `min`, lệch biên của `k`, và dùng **tổng** thay vì **tích** ba đỉnh.

## 2. Đề bài & ví dụ

Bạn có một đa giác lồi `n` cạnh, mỗi đỉnh mang một số nguyên. Cho mảng `values` với `values[i]` là giá trị của đỉnh thứ `i` theo chiều **kim đồng hồ**.

**Tam giác phân** là cách chia đa giác thành các tam giác sao cho **đỉnh của mỗi tam giác cũng là đỉnh của đa giác gốc**, không dùng hình khác tam giác. Cách chia này cho đúng `n - 2` tam giác.

Với mỗi tam giác, **trọng số** là **tích** các giá trị ở ba đỉnh của nó. **Điểm** của tam giác phân là **tổng** trọng số của mọi tam giác.

Trả về **điểm nhỏ nhất** có thể đạt được bởi một tam giác phân của đa giác.

**Ví dụ 1:** `values = [1,2,3]` -> `6`. Đa giác đã là tam giác: `1 * 2 * 3 = 6`.

**Ví dụ 2:** `values = [3,7,4,5]` -> `144`. Có hai tam giác phân: `3*7*5 + 4*5*7 = 245`, hoặc `3*4*5 + 3*4*7 = 144`. Nhỏ nhất là `144`.

**Ví dụ 3:** `values = [1,3,1,4,1,5]` -> `13`.

**Ràng buộc:**

- `n == values.length`
- `3 <= n <= 50`
- `1 <= values[i] <= 100`

In [1]:
from typing import List, Tuple, Optional
from functools import lru_cache
from itertools import combinations, product
import random, time

examples = [
    ([1, 2, 3], 6),
    ([3, 7, 4, 5], 144),
    ([1, 3, 1, 4, 1, 5], 13),
]
for values, expected in examples:
    print(f"values={values} -> kỳ vọng {expected}")

values=[1, 2, 3] -> kỳ vọng 6
values=[3, 7, 4, 5] -> kỳ vọng 144
values=[1, 3, 1, 4, 1, 5] -> kỳ vọng 13


## 3. Kiến thức nền

### 3.1. Tam giác phân và công thức đếm

Tam giác phân đa giác lồi `n` đỉnh dùng đúng `n - 3` đường chéo **đôi một không cắt nhau** (ở trong) và tạo `n - 2` tam giác. Số cách tam giác phân là **số Catalan** `C_{n-2} = C(2(n-2), n-2) / (n - 1)`: `n = 4` có `2`, `n = 5` có `5`, `n = 6` có `14`, `n = 10` có `1430`, và tăng rất nhanh (tới `n = 50` là cỡ `10^26`), nên không thể liệt kê.

In [2]:
from math import comb
def catalan(m): return comb(2 * m, m) // (m + 1)
print({n: catalan(n - 2) for n in (3, 4, 5, 6, 8, 10, 14, 20, 50)})

{3: 1, 4: 2, 5: 5, 6: 14, 8: 132, 10: 1430, 14: 208012, 20: 477638700, 50: 131327898242169365477991900}


### 3.2. Cạnh `(i, j)` thuộc đúng một tam giác

Xét đa giác con có các đỉnh `i, i + 1, ..., j` (cạnh nối `j` về `i` coi là một cạnh của đa giác con). Cạnh `(i, j)` phải thuộc đúng một tam giác của tam giác phân; tam giác đó có đỉnh thứ ba `k` với `i < k < j`. Tam giác `(i, k, j)` tách đa giác con thành hai đa giác nhỏ hơn: `i, ..., k` và `k, ..., j`; hai phần này được tam giác phân **độc lập**. Do đó

`dp[i][j] = min over k in (i, j) of dp[i][k] + dp[k][j] + v[i] * v[k] * v[j]`,

với `dp[i][i + 1] = 0` (một cạnh, không có tam giác). Đáp án `dp[0][n - 1]`.

### 3.3. Thứ tự điền bảng: theo độ dài đoạn

`dp[i][j]` cần `dp[i][k]` và `dp[k][j]` với `i < k < j`: các đoạn **ngắn hơn**. Vậy điền theo **độ dài** `length = j - i` tăng dần (`length = 2, 3, ..., n - 1`). Nếu điền theo `i` tăng dần rồi `j` tăng dần, thì khi tính `dp[i][j]` ô `dp[k][j]` với `k > i` **chưa được tính** (hàng `k` đứng sau hàng `i`).

In [3]:
values = [3, 7, 4, 5]
n = len(values)
dp = [[0] * n for _ in range(n)]
for length in range(2, n):
    for i in range(n - length):
        j = i + length
        dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i + 1, j))
for row in dp:
    print(row)
print("đáp án dp[0][n-1] =", dp[0][n - 1])
assert dp[0][n - 1] == 144

[0, 0, 84, 144]
[0, 0, 0, 140]
[0, 0, 0, 0]
[0, 0, 0, 0]
đáp án dp[0][n-1] = 144


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `values = [3, 7, 4, 5]` (các đỉnh `0, 1, 2, 3`). Cạnh `(0, 3)` thuộc một tam giác `(0, k, 3)`:

- `k = 1`: tam giác `3 * 7 * 5 = 105`, phần còn lại: đa giác con `1, 2, 3` (tam giác `7 * 4 * 5 = 140`) và đa giác con `0, 1` (một cạnh, `0`): tổng `105 + 140 = 245`.
- `k = 2`: tam giác `3 * 4 * 5 = 60`, đa giác con `0, 1, 2` (`3 * 7 * 4 = 84`) và `2, 3` (`0`): tổng `60 + 84 = 144`.

`min(245, 144) = 144`.

Ví dụ 3 `[1, 3, 1, 4, 1, 5]`: tam giác phân tối ưu dùng **tam giác giữa** `(0, 2, 4)` gồm ba đỉnh giá trị `1` (trọng số `1`) rồi ba tam giác "tai" quanh nó: `(0, 1, 2) = 1 * 3 * 1 = 3`, `(2, 3, 4) = 1 * 4 * 1 = 4`, `(0, 4, 5) = 1 * 1 * 5 = 5`. Tổng `3 + 1 + 4 + 5 = 13`. (Cắt "tai" tham lam từng đỉnh nhỏ nhất sẽ không tìm ra cấu trúc này.)

## 5. Cách 1 — Brute Force: mọi tập đường chéo không cắt nhau

**Ý tưởng.** Làm đúng **theo định nghĩa** hình học: một tam giác phân là một tập gồm `n - 3` **đường chéo đôi một không cắt nhau**. Với mỗi tập như vậy, các cạnh (cạnh đa giác và đường chéo đã chọn) tạo một đồ thị mà các **tam giác** của nó (bộ ba đỉnh `a < b < c` với cả ba cạnh `ab`, `bc`, `ac` đều có mặt) chính là `n - 2` tam giác của tam giác phân. Điểm = tổng tích ba đỉnh của các tam giác đó. Không dùng bất kỳ công thức quy hoạch động nào nên đây là **oracle độc lập**. Chỉ dùng cho `n` nhỏ.

In [4]:
def diagonals(n: int) -> List[Tuple[int, int]]:
    return [(i, j) for i in range(n) for j in range(i + 2, n) if not (i == 0 and j == n - 1)]

def cross(d1: Tuple[int, int], d2: Tuple[int, int]) -> bool:
    """Hai đường chéo (a, b), (c, d) của đa giác lồi (đỉnh đánh số 0..n-1) cắt nhau ở phần trong?"""
    (a, b), (c, d) = d1, d2
    return (a < c < b < d) or (c < a < d < b)

def triangulation_brute(values: List[int]) -> int:
    n = len(values)
    diags = diagonals(n)
    best = None
    for chosen in combinations(diags, n - 3):
        if any(cross(d1, d2) for d1, d2 in combinations(chosen, 2)):
            continue
        edges = {(i, (i + 1) % n) if i < (i + 1) % n else ((i + 1) % n, i) for i in range(n)} | set(chosen)
        score, count = 0, 0
        for a, b, c in combinations(range(n), 3):
            if (a, b) in edges and (b, c) in edges and (a, c) in edges:
                score += values[a] * values[b] * values[c]
                count += 1
        assert count == n - 2, (chosen, count)          # đúng n - 2 tam giác
        if best is None or score < best:
            best = score
    return best

for values, expected in examples:
    got = triangulation_brute(values)
    print(f"values={values}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

values=[1, 2, 3]: brute = 6 OK
values=[3, 7, 4, 5]: brute = 144 OK
values=[1, 3, 1, 4, 1, 5]: brute = 13 OK


**Độ phức tạp và nút thắt.** `C(n(n - 3)/2, n - 3)` tập đường chéo (rất lớn), chỉ `Catalan(n - 2)` tập hợp lệ. Với `n = 8` đã `C(20, 5) = 15504` tập phải thử. Đo thử:

In [5]:
rng = random.Random(1039)
for n in (5, 6, 7, 8, 9):
    values = [rng.randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter()
    triangulation_brute(values)
    print(f"n={n}: C({n * (n - 3) // 2},{n - 3}) = {comb(n * (n - 3) // 2, n - 3):>7} tập, {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("n = 50: không thể")

n=5: C(5,2) =      10 tập,       0.0 ms
n=6: C(9,3) =      84 tập,       0.1 ms
n=7: C(14,4) =    1001 tập,       0.8 ms
n=8: C(20,5) =   15504 tập,      10.5 ms


n=9: C(27,6) =  296010 tập,     187.5 ms
n = 50: không thể


Nút thắt: mọi tam giác phân chia sẻ nhiều **đa giác con** chung; ta tính lại chúng hàng triệu lần. **Chia để trị** theo cạnh `(i, j)` và nhớ lời giải của mỗi đa giác con.

## 6. Ý tưởng thuật toán / bước đột phá

**Chọn đỉnh thứ ba `k` của tam giác chứa cạnh `(i, j)`, rồi giải hai đa giác con.**

Công thức ở mục 3.2: `dp[i][j] = min_k dp[i][k] + dp[k][j] + v[i] * v[k] * v[j]`, `dp[i][i + 1] = 0`. Có `O(n^2)` trạng thái `(i, j)`, mỗi trạng thái thử `O(n)` giá trị `k`: `O(n^3)` (với `n = 50`: tối đa `~ 20000` phép).

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy không nhớ theo công thức (số Catalan lời gọi) |
| 3 | Đệ quy có nhớ `lru_cache` |
| 4 | Bảng bottom-up theo độ dài đoạn (nộp) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy không nhớ theo công thức

**Giải thích từng đoạn.** `f(i, j)`: điểm nhỏ nhất của đa giác con `i..j`; `j - i < 2` thì `0` (không đủ ba đỉnh); ngược lại `min` theo `k` của `f(i, k) + f(k, j) + v[i] * v[k] * v[j]`. Không lưu kết quả nên số lời gọi tăng theo số Catalan: chỉ dùng cho `n` nhỏ (làm cầu nối từ định nghĩa sang quy hoạch động).

In [6]:
calls = 0
def triangulation_recursive(values: List[int]) -> int:
    global calls
    calls = 0

    def f(i: int, j: int) -> int:
        global calls
        calls += 1
        if j - i < 2:
            return 0
        return min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))

    return f(0, len(values) - 1)

for values, expected in examples:
    got = triangulation_recursive(values)
    print(f"values={values}: đệ quy không nhớ = {got} ({calls} lời gọi)", "OK" if got == expected else "SAI")
    assert got == expected

values=[1, 2, 3]: đệ quy không nhớ = 6 (3 lời gọi) OK
values=[3, 7, 4, 5]: đệ quy không nhớ = 144 (9 lời gọi) OK
values=[1, 3, 1, 4, 1, 5]: đệ quy không nhớ = 13 (81 lời gọi) OK


**Code nộp LeetCode (Cách 2):** (đúng nhưng **quá chậm** với `n` lớn; chỉ minh họa)

In [7]:
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        def f(i: int, j: int) -> int:
            if j - i < 2:
                return 0
            return min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))

        return f(0, len(values) - 1)

sol_recursive = Solution().minScoreTriangulation
for values, expected in examples:
    assert sol_recursive(values) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** số lời gọi tăng theo hàm mũ (xấp xỉ `3^n` hoặc nhiều hơn). Đo thử:

In [8]:
for n in (4, 6, 8, 10, 12, 13):
    values = [random.Random(n).randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter()
    triangulation_recursive(values)
    print(f"n={n:2}: {calls:9} lời gọi, {(time.perf_counter() - t0) * 1000:9.1f} ms")

n= 4:         9 lời gọi,       0.0 ms
n= 6:        81 lời gọi,       0.0 ms
n= 8:       729 lời gọi,       0.2 ms
n=10:      6561 lời gọi,       1.6 ms
n=12:     59049 lời gọi,      14.2 ms
n=13:    177147 lời gọi,      43.0 ms


**Giải thích tăng trưởng.** Mỗi lời gọi `f(i, j)` sinh `j - i - 1` cặp lời gọi con; không nhớ nên các đa giác con giống nhau bị giải lại nhiều lần. Nhớ lại là xong.

---

### 7.2. Cách 3 — Đệ quy có nhớ (`lru_cache`)

**Giải thích từng đoạn.** Giống Cách 2, nhưng thêm `@lru_cache(None)` cho `f(i, j)`: mỗi `(i, j)` chỉ tính một lần. Có `O(n^2)` trạng thái, mỗi trạng thái `O(n)`. Độ sâu đệ quy tối đa `n` (`<= 50`).

In [9]:
def triangulation_memo(values: List[int]) -> int:
    @lru_cache(None)
    def f(i: int, j: int) -> int:
        if j - i < 2:
            return 0
        return min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))

    return f(0, len(values) - 1)

for values, expected in examples:
    got = triangulation_memo(values)
    print(f"values={values}: có nhớ = {got}", "OK" if got == expected else "SAI")
    assert got == expected

values=[1, 2, 3]: có nhớ = 6 OK
values=[3, 7, 4, 5]: có nhớ = 144 OK
values=[1, 3, 1, 4, 1, 5]: có nhớ = 13 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        from functools import lru_cache

        @lru_cache(None)
        def f(i: int, j: int) -> int:
            if j - i < 2:                       # một cạnh: không có tam giác
                return 0
            return min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))

        return f(0, len(values) - 1)

sol_memo = Solution().minScoreTriangulation
for values, expected in examples:
    assert sol_memo(values) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n^3)`, bộ nhớ `O(n^2)` (bảng nhớ) cộng `O(n)` ngăn xếp đệ quy.

---

### 7.3. Cách 4 — Bảng bottom-up theo độ dài (cách nộp chính)

**Giải thích từng đoạn.**

- `dp = [[0] * n for _ in range(n)]`: ban đầu mọi ô `0` (các đoạn `j - i < 2` đúng là `0`).
- `for length in range(2, n)`: độ dài đoạn `j - i` từ `2` đến `n - 1`.
- `for i in range(n - length)`: `j = i + length`.
- `dp[i][j]`: **đặt lại** thành `min` theo `k` (không dùng `0` ban đầu để so sánh).

In [11]:
def triangulation_table(values: List[int]) -> int:
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            best = float("inf")
            for k in range(i + 1, j):
                best = min(best, dp[i][k] + dp[k][j] + values[i] * values[k] * values[j])
            dp[i][j] = best
    return dp[0][n - 1]

for values, expected in examples:
    got = triangulation_table(values)
    print(f"values={values}: bảng = {got}", "OK" if got == expected else "SAI")
    assert got == expected

values=[1, 2, 3]: bảng = 6 OK
values=[3, 7, 4, 5]: bảng = 144 OK
values=[1, 3, 1, 4, 1, 5]: bảng = 13 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        n = len(values)
        dp = [[0] * n for _ in range(n)]            # dp[i][j]: điểm nhỏ nhất của đa giác con i..j (dp[i][i+1] = 0)
        for length in range(2, n):                   # độ dài đoạn tăng dần: dp[i][k], dp[k][j] đã sẵn
            for i in range(n - length):
                j = i + length
                dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j]
                               for k in range(i + 1, j))
        return dp[0][n - 1]

sol_table = Solution().minScoreTriangulation
for values, expected in examples:
    assert sol_table(values) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n^3)` (`n = 50`: `~ 20000` phép), bộ nhớ `O(n^2)`.

## 8. TRACE từng bước

### 8.1. Bảng `dp` cho `[1, 3, 1, 4, 1, 5]`

In bảng sau khi điền theo từng độ dài, kèm `k` tối ưu.

In [13]:
def trace_table(values):
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            options = [(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j], k) for k in range(i + 1, j)]
            dp[i][j], bestk = min(options)
            print(f"dp[{i}][{j}] (độ dài {length}): các lựa chọn (điểm, k) = {options} -> {dp[i][j]} với k = {bestk}")
    return dp[0][n - 1]

assert trace_table([1, 3, 1, 4, 1, 5]) == 13

dp[0][2] (độ dài 2): các lựa chọn (điểm, k) = [(3, 1)] -> 3 với k = 1
dp[1][3] (độ dài 2): các lựa chọn (điểm, k) = [(12, 2)] -> 12 với k = 2
dp[2][4] (độ dài 2): các lựa chọn (điểm, k) = [(4, 3)] -> 4 với k = 3
dp[3][5] (độ dài 2): các lựa chọn (điểm, k) = [(20, 4)] -> 20 với k = 4
dp[0][3] (độ dài 3): các lựa chọn (điểm, k) = [(24, 1), (7, 2)] -> 7 với k = 2
dp[1][4] (độ dài 3): các lựa chọn (điểm, k) = [(7, 2), (24, 3)] -> 7 với k = 2
dp[2][5] (độ dài 3): các lựa chọn (điểm, k) = [(40, 3), (9, 4)] -> 9 với k = 4
dp[0][4] (độ dài 4): các lựa chọn (điểm, k) = [(10, 1), (8, 2), (11, 3)] -> 8 với k = 2
dp[1][5] (độ dài 4): các lựa chọn (điểm, k) = [(24, 2), (92, 3), (22, 4)] -> 22 với k = 4
dp[0][5] (độ dài 5): các lựa chọn (điểm, k) = [(37, 1), (17, 2), (47, 3), (13, 4)] -> 13 với k = 4


### 8.2. Cạnh `(0, n - 1)` của ví dụ 2: hai tam giác phân

Liệt kê hai tam giác phân và điểm của mỗi cách (khớp lời giải tay ở mục 4).

In [14]:
values = [3, 7, 4, 5]
for k in (1, 2):
    triangles = [(0, k, 3)] + ([(1, 2, 3)] if k == 1 else [(0, 1, 2)])
    score = sum(values[a] * values[b] * values[c] for a, b, c in triangles)
    print(f"k = {k}: các tam giác {triangles} -> điểm {score}")
assert min(3 * 7 * 5 + 7 * 4 * 5, 3 * 4 * 5 + 3 * 7 * 4) == 144

k = 1: các tam giác [(0, 1, 3), (1, 2, 3)] -> điểm 245
k = 2: các tam giác [(0, 2, 3), (0, 1, 2)] -> điểm 144


### 8.3. Thứ tự điền bảng: vì sao phải theo độ dài

Nếu điền theo `i` tăng dần trước, ô `dp[k][j]` (hàng `k > i`) còn bằng `0` khi tính `dp[i][j]`, làm điểm bị đánh giá **thấp hơn** thực tế.

In [15]:
def wrong_order(values):
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for i in range(n):                              # sai thứ tự: hàng i trước hàng k > i
        for j in range(i + 2, n):
            dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i + 1, j))
    return dp[0][n - 1]

print("[3,7,4,5]: sai ->", wrong_order([3, 7, 4, 5]), "| đúng ->", Solution().minScoreTriangulation([3, 7, 4, 5]))
assert wrong_order([1, 3, 1, 4, 1, 5]) != 13

[3,7,4,5]: sai -> 105 | đúng -> 144


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (bảng bottom-up theo độ dài): `O(n^3)`, không đệ quy.

```python
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        n = len(values)
        dp = [[0] * n for _ in range(n)]            # dp[i][j]: điểm nhỏ nhất của đa giác con i..j (dp[i][i+1] = 0)
        for length in range(2, n):                   # độ dài đoạn tăng dần: dp[i][k], dp[k][j] đã sẵn
            for i in range(n - length):
                j = i + length
                dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j]
                               for k in range(i + 1, j))
        return dp[0][n - 1]
```

In [16]:
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        n = len(values)
        dp = [[0] * n for _ in range(n)]            # dp[i][j]: điểm nhỏ nhất của đa giác con i..j (dp[i][i+1] = 0)
        for length in range(2, n):                   # độ dài đoạn tăng dần: dp[i][k], dp[k][j] đã sẵn
            for i in range(n - length):
                j = i + length
                dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j]
                               for k in range(i + 1, j))
        return dp[0][n - 1]

sol = Solution()
for values, expected in examples:
    got = sol.minScoreTriangulation(values)
    print(f"values={values}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

values=[1, 2, 3]: 6 OK
values=[3, 7, 4, 5]: 144 OK
values=[1, 3, 1, 4, 1, 5]: 13 OK


## 10. Vì sao đúng

**Cấu trúc.** Trong một tam giác phân của đa giác con `P(i, j)` (các đỉnh `i, i + 1, ..., j`, với cạnh đóng `(j, i)`), cạnh `(i, j)` thuộc **đúng một** tam giác `(i, k, j)` với `i < k < j` (đỉnh thứ ba nằm trong đoạn đỉnh). Tam giác này chia `P(i, j)` thành `P(i, k)`, `P(k, j)` và chính nó; các tam giác còn lại thuộc hoặc `P(i, k)` hoặc `P(k, j)`, và tam giác phân của hai phần được chọn **độc lập** (không có cạnh chung ảnh hưởng đến trọng số vì trọng số chỉ phụ thuộc đỉnh của từng tam giác). Ngược lại ghép một tam giác phân của `P(i, k)`, một của `P(k, j)` và tam giác `(i, k, j)` cho một tam giác phân của `P(i, j)`.

**Công thức.** Do đó `dp[i][j] = min_k (dp[i][k] + dp[k][j] + v[i] * v[k] * v[j])`, với nền `dp[i][i + 1] = 0`. Đúng bằng quy nạp theo độ dài `j - i`. Số tam giác của tam giác phân đa giác `P(i, j)` là `j - i - 1`, và tổng thể `n - 2`.

**Thứ tự.** Điền theo độ dài tăng dần bảo đảm `dp[i][k]`, `dp[k][j]` (độ dài nhỏ hơn) đã có.

**Tham lam sai.** Cắt đỉnh nhỏ nhất (tai nhỏ nhất) ở mỗi bước **không** tối ưu: `[3, 7, 4, 5]` cắt đỉnh `3` trước cho `3 * 5 * 7 = 105` và còn tam giác `7 * 4 * 5 = 140`: tổng `245 > 144`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi tập `n - 3` đường chéo không cắt nhau và tính tổng trên các tam giác thực sự (bộ ba đỉnh có đủ ba cạnh). Ta đối chiếu ba cách còn lại trên **mọi** mảng giá trị `1..3` độ dài `3..6`, rồi trên nhiều mảng ngẫu nhiên có seed (`n` tới `7` so với oracle, `n` tới `50` so các cách nhanh).

In [17]:
total = 0
for n in range(3, 7):
    for values in product((1, 2, 3), repeat=n):
        v = list(values)
        ref = triangulation_brute(v)
        assert triangulation_memo(v) == triangulation_table(v) == sol.minScoreTriangulation(v) == ref, v
        if n <= 6:
            assert triangulation_recursive(v) == ref
        total += 1
print("đã đối chiếu", total, "mảng (giá trị 1..3, n = 3..6): bốn cách khớp oracle hình học")

đã đối chiếu 1080 mảng (giá trị 1..3, n = 3..6): bốn cách khớp oracle hình học


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập đường chéo | `O(C(n(n-3)/2, n-3) * n^3)` | `O(n)` |
| 2. Đệ quy không nhớ | hàm mũ (số Catalan lời gọi) | `O(n)` |
| 3. Đệ quy có nhớ | `O(n^3)` | `O(n^2)` |
| 4. Bảng bottom-up | `O(n^3)` | `O(n^2)` |

Đo ở cỡ tối đa (`n = 50`) với các dạng giá trị: ngẫu nhiên, toàn `100`, tăng dần, đỉnh nhỏ nằm xen kẽ:

In [18]:
rng = random.Random(11)
cases = {"ngẫu nhiên 1..100": [rng.randint(1, 100) for _ in range(50)], "toàn 100": [100] * 50,
         "tăng dần 1..50": list(range(1, 51)), "xen kẽ 1, 100": [1 if i % 2 == 0 else 100 for i in range(50)]}
for label, values in cases.items():
    ref = triangulation_table(values)
    row = [f"{label:20}"]
    for name, fn in (("có nhớ", sol_memo), ("bảng", sol_table), ("nộp", sol.minScoreTriangulation)):
        t0 = time.perf_counter()
        got = fn(values)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên 1..100    | có nhớ   10.00 ms | bảng    5.00 ms | nộp    4.92 ms | đáp án 281590


toàn 100             | có nhớ   10.48 ms | bảng    4.79 ms | nộp    4.99 ms | đáp án 48000000


tăng dần 1..50       | có nhớ   11.43 ms | bảng    4.55 ms | nộp    4.43 ms | đáp án 41648
xen kẽ 1, 100        | có nhớ    9.88 ms | bảng    4.37 ms | nộp    4.36 ms | đáp án 2523


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (`n` tới `50`) và các tính chất. Đáp án là một số **duy nhất**.

In [19]:
def all_solutions():
    return {"memo": triangulation_memo, "table": triangulation_table, "sol_memo": sol_memo, "sol_table": sol_table, "final": sol.minScoreTriangulation}

def check_all(values, expected=None, brute=True):
    results = {name: fn(list(values)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (values, results)
    ref = results["table"]
    if brute and len(values) <= 7:
        assert triangulation_brute(values) == ref, (values, triangulation_brute(values), ref)
    if len(values) <= 9:
        assert triangulation_recursive(values) == ref
    if expected is not None:
        assert ref == expected, (values, ref, expected)
    return ref

# 12.1 ví dụ đề
for values, expected in examples:
    check_all(values, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1, 1, 1], 1),                                    # tam giác: tích 1
    ([100, 100, 100], 10 ** 6),                        # tam giác biên: 100^3
    ([2, 3, 4], 24),                                   # tam giác: 2 * 3 * 4
    ([1, 1, 1, 1], 2),                                 # tứ giác toàn 1: hai tam giác trọng số 1
    ([2, 2, 2, 2], 16),                                # tứ giác toàn 2: hai tam giác trọng số 8
    ([3, 7, 4, 5], 144),                               # ví dụ 2
    ([5, 4, 3, 2], None),                              # tứ giác bất kỳ: để oracle quyết định
    ([1, 3, 1, 4, 1, 5], 13),                          # ví dụ 3
    ([100] * 5, 3 * 10 ** 6),                          # ngũ giác toàn 100: 3 tam giác * 10^6
    ([1, 100, 1, 100, 1, 100], None),                  # xen kẽ: để oracle quyết định
    ([1] * 50, 48),                                    # n = 50, toàn 1: 48 tam giác trọng số 1
    ([100] * 50, 48 * 10 ** 6),                        # n = 50, toàn 100: 48 tam giác trọng số 10^6
]
for values, expected in edge:
    check_all(values, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_greedy_ear(values):                        # tham lam: liên tục cắt đỉnh có giá trị nhỏ nhất
    v = list(values)
    total = 0
    while len(v) > 2:
        i = v.index(min(v))
        total += v[i - 1] * v[i] * v[(i + 1) % len(v)]
        v.pop(i)
    return total

def wrong_order_loops(values):                       # lặp i tăng dần trước (dùng ô chưa tính)
    return wrong_order(values)

def wrong_init_zero(values):                         # min với 0 ban đầu: dp[i][j] = min(dp[i][j], ...) khi dp khởi tạo 0
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            for k in range(i + 1, j):
                dp[i][j] = min(dp[i][j], dp[i][k] + dp[k][j] + values[i] * values[k] * values[j])
    return dp[0][n - 1]

def wrong_sum_instead_of_product(values):            # tổng thay vì tích ba đỉnh
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] + values[k] + values[j] for k in range(i + 1, j))
    return dp[0][n - 1]

def wrong_k_range(values):                           # k chạy từ i (thay vì i + 1): thêm tam giác suy biến
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i, j))
    return dp[0][n - 1]

def find_counter(wrong, vmax=4, max_n=6):
    for n in range(3, max_n + 1):
        for values in product(range(1, vmax + 1), repeat=n):
            v = list(values)
            if wrong(v) != triangulation_brute(v):
                return v
    return None

counters = {}
for name, fn in (("tham lam cắt đỉnh nhỏ nhất", wrong_greedy_ear), ("thứ tự lặp sai (i trước)", wrong_order_loops),
                 ("khởi tạo dp bằng 0 rồi lấy min", wrong_init_zero), ("dùng tổng thay vì tích", wrong_sum_instead_of_product),
                 ("k bắt đầu từ i", wrong_k_range)):
    v = find_counter(fn)
    assert v is not None, name
    counters[name] = v
    print(f"{name:36} values={v}: sai -> {fn(v)}, đúng -> {triangulation_brute(v)}")
    check_all(v)
print("12.3 phản ví dụ: OK")

tham lam cắt đỉnh nhỏ nhất           values=[1, 1, 1, 2]: sai -> 4, đúng -> 3
thứ tự lặp sai (i trước)             values=[1, 1, 1, 1]: sai -> 1, đúng -> 2
khởi tạo dp bằng 0 rồi lấy min       values=[1, 1, 1]: sai -> 0, đúng -> 1
dùng tổng thay vì tích               values=[1, 1, 1]: sai -> 3, đúng -> 1
k bắt đầu từ i                       values=[1, 2, 1]: sai -> 1, đúng -> 2
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: n nhỏ so với oracle hình học, n tới 50 so các cách nhanh (đệ quy có nhớ vs bảng), nhiều miền giá trị
rng = random.Random(4039)
for _ in range(300):
    n = rng.randint(3, 7)
    check_all([rng.randint(1, rng.choice([2, 5, 100])) for _ in range(n)])
for _ in range(40):
    n = rng.randint(8, 50)
    v = [rng.randint(1, rng.choice([3, 10, 100])) for _ in range(n)]
    check_all(v, brute=False)
print("12.4 ngẫu nhiên: OK")

# 12.5 tính chất: bất biến theo xoay vòng và đảo chiều đa giác; tăng một giá trị không làm giảm điểm; nhân đôi mọi giá trị nhân điểm với 8; chặn dưới và chặn trên
rng = random.Random(5039)
for _ in range(300):
    n = rng.randint(3, 20)
    v = [rng.randint(1, 100) for _ in range(n)]
    r = sol.minScoreTriangulation(v)
    s = rng.randrange(n)
    assert sol.minScoreTriangulation(v[s:] + v[:s]) == r                 # xoay vòng
    assert sol.minScoreTriangulation(v[::-1]) == r                       # đảo chiều
    w = list(v); w[rng.randrange(n)] += rng.randint(1, 10)
    assert sol.minScoreTriangulation(w) >= r                             # đơn điệu
    assert sol.minScoreTriangulation([2 * x for x in v]) == 8 * r        # mỗi trọng số nhân 8
    assert (n - 2) * min(v) ** 3 <= r <= (n - 2) * max(v) ** 3           # chặn dưới/trên
print("12.5 tính chất: OK")

12.4 ngẫu nhiên: OK
12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn cắt đỉnh nhỏ nhất

Cắt "tai" tại đỉnh nhỏ nhất rất hấp dẫn nhưng **không** tối ưu vì nó chọn tam giác với hai đỉnh kề, bỏ lỡ các tam giác dùng đỉnh nhỏ chung nhiều lần.

In [22]:
v = counters["tham lam cắt đỉnh nhỏ nhất"]
print("values =", v, "| sai ->", wrong_greedy_ear(v), "| đúng ->", Solution().minScoreTriangulation(v))
assert wrong_greedy_ear(v) != Solution().minScoreTriangulation(v)
print("ví dụ 2: sai ->", wrong_greedy_ear([3, 7, 4, 5]), "| đúng ->", Solution().minScoreTriangulation([3, 7, 4, 5]))

values = [1, 1, 1, 2] | sai -> 4 | đúng -> 3
ví dụ 2: sai -> 245 | đúng -> 144


Vì sao sai: ở ví dụ 2 cắt đỉnh `3` trước cho `245` thay vì `144`. **Đúng:** quy hoạch động xét mọi đỉnh `k` của tam giác chứa cạnh `(i, j)`.

### Lỗi 2 — Thứ tự lặp sai

Điền theo `i` tăng dần trước khi các hàng sau (`k > i`) được tính làm dùng ô `0`.

In [23]:
v = counters["thứ tự lặp sai (i trước)"]
print("values =", v, "| sai ->", wrong_order_loops(v), "| đúng ->", Solution().minScoreTriangulation(v))
assert wrong_order_loops(v) != Solution().minScoreTriangulation(v)

values = [1, 1, 1, 1] | sai -> 1 | đúng -> 2


Vì sao sai: `dp[k][j]` với `k > i` chưa được tính khi tới hàng `i`. **Đúng:** lặp theo độ dài `j - i` tăng dần.

### Lỗi 3 — Khởi tạo `dp` bằng `0` rồi lấy `min`

Nếu `dp[i][j]` đã là `0` mà ta lấy `min(dp[i][j], ...)` thì `min` luôn giữ `0`.

In [24]:
v = counters["khởi tạo dp bằng 0 rồi lấy min"]
print("values =", v, "| sai ->", wrong_init_zero(v), "| đúng ->", Solution().minScoreTriangulation(v))
assert wrong_init_zero(v) != Solution().minScoreTriangulation(v)

values = [1, 1, 1] | sai -> 0 | đúng -> 1


Vì sao sai: `0` nhỏ hơn mọi ứng viên dương. **Đúng:** khởi tạo `+inf` cho ô cần lấy `min`, hoặc tính `min` trực tiếp bằng `min(... for k in ...)`.

### Lỗi 4 — Dùng tổng thay vì tích ba đỉnh

Trọng số tam giác là **tích** các giá trị ở ba đỉnh.

In [25]:
v = counters["dùng tổng thay vì tích"]
print("values =", v, "| sai ->", wrong_sum_instead_of_product(v), "| đúng ->", Solution().minScoreTriangulation(v))
assert wrong_sum_instead_of_product(v) != Solution().minScoreTriangulation(v)

values = [1, 1, 1] | sai -> 3 | đúng -> 1


Vì sao sai: ví dụ 1 `[1, 2, 3]` cho `1 + 2 + 3 = 6` trùng một cách tình cờ nhưng các ví dụ khác sai. **Đúng:** `values[i] * values[k] * values[j]`.

### Lỗi 5 — Lệch biên của `k`

`k` phải nằm **thực sự giữa** `i` và `j` (`i < k < j`). Cho `k = i` tạo "tam giác" suy biến `(i, i, j)` với trọng số dương không tồn tại.

In [26]:
v = counters["k bắt đầu từ i"]
print("values =", v, "| sai ->", wrong_k_range(v), "| đúng ->", Solution().minScoreTriangulation(v))
assert wrong_k_range(v) != Solution().minScoreTriangulation(v)

values = [1, 2, 1] | sai -> 1 | đúng -> 2


Vì sao sai: `k = i` làm `dp[i][i] + dp[i][j] + ...` tự tham chiếu ô đang tính. **Đúng:** `range(i + 1, j)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia đa giác/đoạn thành các phần, chi phí mỗi phần phụ thuộc đầu mút" ⇒ **quy hoạch động đoạn** `dp[i][j]`, thử điểm chia `k` giữa `i` và `j`.
- Thứ tự điền: theo **độ dài** đoạn tăng dần, hoặc `i` giảm dần và `j` tăng dần.
- Nền: đoạn không đủ phần tử để tạo đơn vị chi phí có giá `0`.
- Cùng họ: Burst Balloons (312), Minimum Cost to Merge Stones (1000), Matrix Chain Multiplication, Minimum Cost Tree From Leaf Values (1130), Strange Printer (664).
- Test nên có: `n = 3`, `n = 4`, mọi giá trị bằng nhau, các giá trị khác biệt lớn, `n = 50`.
- Nhận diện tham lam sai bằng cách thử `[3, 7, 4, 5]`.

## 15. Biến thể

1. **Điểm lớn nhất:** thay `min` bằng `max` (cùng công thức).
2. **Trọng số khác:** tổng ba đỉnh, hay `max` ba đỉnh: chỉ đổi hệ số `v[i] * v[k] * v[j]`.
3. **In một tam giác phân tối ưu:** lưu `k` tối ưu rồi truy vết (bài tập 5).
4. Bài liên quan: 312, 1130, 1000.

Biến thể 1 (điểm lớn nhất) kiểm bằng oracle hình học:

In [27]:
def triangulation_max_table(values):
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            dp[i][j] = max(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i + 1, j))
    return dp[0][n - 1]

def triangulation_max_brute(values):
    n = len(values)
    best = None
    for chosen in combinations(diagonals(n), n - 3):
        if any(cross(d1, d2) for d1, d2 in combinations(chosen, 2)):
            continue
        edges = {tuple(sorted((i, (i + 1) % n))) for i in range(n)} | set(chosen)
        score = sum(values[a] * values[b] * values[c] for a, b, c in combinations(range(n), 3)
                    if (a, b) in edges and (b, c) in edges and (a, c) in edges)
        best = score if best is None else max(best, score)
    return best

rng = random.Random(15)
for _ in range(100):
    v = [rng.randint(1, 20) for _ in range(rng.randint(3, 7))]
    assert triangulation_max_table(v) == triangulation_max_brute(v)
print("[3, 7, 4, 5] -> max", triangulation_max_table([3, 7, 4, 5]), "(= 245), min", Solution().minScoreTriangulation([3, 7, 4, 5]), "(= 144)")

[3, 7, 4, 5] -> max 245 (= 245), min 144 (= 144)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập đường chéo | Theo định nghĩa hình học | rất lớn | `O(n)` | Oracle, `n <= 8` |
| 2. Đệ quy không nhớ | Công thức, không lưu | hàm mũ | `O(n)` | Cầu nối tới DP, `n <= 13` |
| 3. Đệ quy có nhớ | `lru_cache` | `O(n^3)` | `O(n^2)` | Dễ viết |
| 4. Bảng bottom-up | Theo độ dài đoạn | `O(n^3)` | `O(n^2)` | **Cách nộp chính** |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(catalan(1), catalan(2), catalan(3), catalan(4))
print(Solution().minScoreTriangulation([2, 3, 4]), Solution().minScoreTriangulation([1, 1, 1, 1]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn số
my_guess_2 = None      # hai số

### Bài 2 — Tính tay (dễ-vừa)

`values = [1, 3, 1, 4, 1, 5]`. Ghi đáp án (điểm nhỏ nhất). Gợi ý: thử các tam giác phân dùng chung đỉnh nhỏ.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành bảng bottom-up.

In [30]:
def exercise_3(values: List[int]) -> int:
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            # TODO 1: dp[i][j] = min theo k trong range(i + 1, j) của dp[i][k] + dp[k][j] + tích ba đỉnh
            pass
    # TODO 2: trả về dp[0][n - 1]
    pass


# for values, expected in examples:
#     assert exercise_3(values) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(values)` bằng **đệ quy có nhớ** (`lru_cache`).

In [31]:
def exercise_4(values: List[int]) -> int:
    # TODO
    pass


# for values, expected in examples:
#     assert exercise_4(values) == expected

### Bài 5 — Mini challenge: dựng một tam giác phân tối ưu (khó)

Viết `exercise_5(values)` trả về **một danh sách `n - 2` tam giác** `(a, b, c)` (với `a < b < c` là chỉ số đỉnh) tạo thành một tam giác phân có điểm **bằng** điểm nhỏ nhất. Có thể có nhiều đáp án; ta kiểm tra bằng **hàm kiểm tra**: đủ `n - 2` tam giác, mỗi cạnh của đa giác thuộc **đúng một** tam giác, mỗi đường chéo thuộc **đúng hai** hoặc **không** tam giác nào (đúng hơn: nếu thuộc thì thuộc đúng hai), và tổng tích bằng đáp án.

```python
exercise_5([1, 2, 3])        -> [(0, 1, 2)]
exercise_5([3, 7, 4, 5])     -> [(0, 2, 3), (0, 1, 2)]  (điểm 144)
```
Gợi ý: lưu `k` tối ưu trong bảng `choice[i][j]` rồi truy vết đệ quy từ `(0, n - 1)`.

In [32]:
def exercise_5(values: List[int]) -> List[Tuple[int, int, int]]:
    # TODO
    pass


def check_triangulation(values, triangles) -> bool:
    n = len(values)
    if len(triangles) != n - 2:
        return False
    cnt = {}
    for a, b, c in triangles:
        if not (0 <= a < b < c < n):
            return False
        for e in ((a, b), (b, c), (a, c)):
            cnt[e] = cnt.get(e, 0) + 1
    for i in range(n):
        e = tuple(sorted((i, (i + 1) % n)))
        if cnt.get(e, 0) != 1:                         # cạnh đa giác thuộc đúng một tam giác
            return False
    for e, c in cnt.items():
        is_side = e[1] - e[0] == 1 or (e[0] == 0 and e[1] == n - 1)
        if not is_side and c != 2:                      # đường chéo thuộc đúng hai tam giác
            return False
    return sum(values[a] * values[b] * values[c] for a, b, c in triangles) == Solution().minScoreTriangulation(values)


# for values, _ in examples:
#     assert check_triangulation(values, exercise_5(values))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `catalan(m) = C(2m, m) / (m + 1)`.
- Mức 2: `catalan(1) = 1`, `catalan(2) = 2`, `catalan(3) = 5`, `catalan(4) = 14`.
- Mức 3: dòng 1: `1 2 5 14`; dòng 2: `24 2` (`[1,1,1,1]`: hai tam giác trọng số `1`).

**Bài 2**
- Mức 1: các đỉnh `1` ở vị trí `0`, `2`, `4`.
- Mức 2: thử tam giác giữa `(0, 2, 4)` gồm ba đỉnh `1` (trọng số `1`), rồi các tam giác tai `(0, 1, 2)`, `(2, 3, 4)`, `(0, 4, 5)`.
- Mức 3: `3 + 4 + 1 + 5 = 13`.

**Bài 3**
- Mức 1: `dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i + 1, j))`.
- Mức 2: lặp theo `length` từ `2` đến `n - 1`.
- Mức 3: `return dp[0][n - 1]`.

**Bài 4**
- Mức 1: `f(i, j)` trả `0` nếu `j - i < 2`.
- Mức 2: `min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))`.
- Mức 3: `@lru_cache(None)`, `return f(0, n - 1)`.

**Bài 5**
- Mức 1: bảng `dp` và `choice[i][j]` = `k` cho giá trị nhỏ nhất.
- Mức 2: `build(i, j)`: nếu `j - i < 2` thì không có tam giác; ngược lại thêm `(i, k, j)` và gọi `build(i, k)`, `build(k, j)`.
- Mức 3: trả danh sách các tam giác.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(catalan(1), catalan(2), catalan(3), catalan(4))
print(Solution().minScoreTriangulation([2, 3, 4]), Solution().minScoreTriangulation([1, 1, 1, 1]))

assert (catalan(1), catalan(2), catalan(3), catalan(4)) == (1, 2, 5, 14)
assert (Solution().minScoreTriangulation([2, 3, 4]), Solution().minScoreTriangulation([1, 1, 1, 1])) == (24, 2)

1 2 5 14
24 2


### Đáp án Bài 2

Tam giác phân tối ưu của `[1, 3, 1, 4, 1, 5]` gồm các tam giác `(0, 1, 2) = 3`, `(2, 3, 4) = 4`, `(0, 2, 4) = 1`, `(0, 4, 5) = 5`: tổng `13` (cũng được oracle hình học và bảng quy hoạch động xác nhận).

In [34]:
hand_answer_solution = 13
assert Solution().minScoreTriangulation([1, 3, 1, 4, 1, 5]) == hand_answer_solution == triangulation_brute([1, 3, 1, 4, 1, 5])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(values: List[int]) -> int:
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j] for k in range(i + 1, j))
    return dp[0][n - 1]

for values, expected in examples:
    assert exercise_3_solution(values) == expected
rng = random.Random(3)
for _ in range(200):
    v = [rng.randint(1, 20) for _ in range(rng.randint(3, 7))]
    assert exercise_3_solution(v) == triangulation_brute(v)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(values: List[int]) -> int:
    @lru_cache(None)
    def f(i: int, j: int) -> int:
        if j - i < 2:
            return 0
        return min(f(i, k) + f(k, j) + values[i] * values[k] * values[j] for k in range(i + 1, j))

    return f(0, len(values) - 1)

for values, expected in examples:
    assert exercise_4_solution(values) == expected
rng = random.Random(4)
for _ in range(200):
    v = [rng.randint(1, 20) for _ in range(rng.randint(3, 7))]
    assert exercise_4_solution(v) == triangulation_brute(v)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(values: List[int]) -> List[Tuple[int, int, int]]:
    n = len(values)
    dp = [[0] * n for _ in range(n)]
    choice = [[-1] * n for _ in range(n)]
    for length in range(2, n):
        for i in range(n - length):
            j = i + length
            best, bestk = None, -1
            for k in range(i + 1, j):
                c = dp[i][k] + dp[k][j] + values[i] * values[k] * values[j]
                if best is None or c < best:
                    best, bestk = c, k
            dp[i][j], choice[i][j] = best, bestk
    triangles = []
    stack = [(0, n - 1)]
    while stack:
        i, j = stack.pop()
        if j - i < 2:
            continue
        k = choice[i][j]
        triangles.append((i, k, j))
        stack.append((i, k))
        stack.append((k, j))
    return triangles

for values, _ in examples:
    assert check_triangulation(values, exercise_5_solution(values))
rng = random.Random(5)
for _ in range(300):
    v = [rng.randint(1, 20) for _ in range(rng.randint(3, 12))]
    assert check_triangulation(v, exercise_5_solution(v)), v
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (quy hoạch động đoạn)

```python
class Solution:
    def minScoreTriangulation(self, values: List[int]) -> int:
        n = len(values)
        dp = [[0] * n for _ in range(n)]            # dp[i][j]: điểm nhỏ nhất của đa giác con i..j
        for length in range(2, n):                   # độ dài đoạn tăng dần
            for i in range(n - length):
                j = i + length
                dp[i][j] = min(dp[i][k] + dp[k][j] + values[i] * values[k] * values[j]
                               for k in range(i + 1, j))
        return dp[0][n - 1]
```

Công thức nhớ nhanh: `dp[i][j] = min_k (dp[i][k] + dp[k][j] + v[i] * v[k] * v[j])` với `i < k < j`, nền `dp[i][i + 1] = 0`.

### Checklist trước khi submit

- [ ] Điền bảng theo **độ dài** đoạn tăng dần (không theo `i` rồi `j` thuần)?
- [ ] `k` chạy trong `range(i + 1, j)` (nằm thực sự giữa `i` và `j`)?
- [ ] Trọng số là **tích** `values[i] * values[k] * values[j]`?
- [ ] Không khởi tạo `0` rồi lấy `min` (dùng `min(... for k ...)` trực tiếp hoặc `+inf`)?
- [ ] Không dùng tham lam "cắt đỉnh nhỏ nhất"?
- [ ] Đã thử: `n = 3`, `n = 4` (`[3,7,4,5]`), toàn giá trị bằng nhau, `n = 50`?